## LoRA adapters for PEFT

Source: https://thinkingmachines.ai/blog/lora/

How to use LoRA reliably to match FullFT

- learning rate for LoRA is higher by a factor of 10 compared to high rank LoRAs (d_embed=512)
- larger batch size degrades LoRA performance
- LoRA should be placed both on MLP and attention layers

Notes on how LoRA works

- since the weights are parameterized by a matmul - the gradients that flow back into A, B (LoRA matrices) have magnitudes dependent on each other
  - as B is initialized at zero, LoRA needs more steps to warm up (hence higher LR)
  - as magnitudes of B, A grows, the effective learning rate grows as well
- bigger batch sizes hurt because they
  1.  reduce training steps which is not good for LoRA as mentioned above
  2.  bigger steps mean bigger gradients which can blow up the LR later on as well (since the effective LR is tied to A, B's magnitude as well)

RsLora

- original LoRA paper (Hu et. al, 2021) proposes the weighting of the LoRA update should be $\dfrac{\alpha}{r}$
- RsLora (Kalajdzievski, 2023) proposes that the actual learning rate should be $\dfrac{\alpha}{\sqrt{r}}$ given that we are doing a dot product over $r$ terms, each with variance $\dfrac{1}{d}$, thus the std is the RsLora proposed weight
  - previously LoRA used to perform worse or the same on higher ranks (i.e. rank=8 vs rank=256), this was because the gradients were being suppressed by a learning rate that was $\sqrt{r}$ too high

Other things covered

- practice loading a pre-trained model in
- qLora (NVFP4 for the base model and bf16 for Lora adapter)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from abc import abstractmethod
from einops import rearrange

# initialize a base LoRA abstract class
class LoraBase(nn.Module):
	
	def __init__(self, in_dim:int, out_dim:int, rank:int=8, lora_dropout:float=0.1, lora_alpha:float=0.1, use_rslora:bool=True):
		# NOTE: no nn.Module.__init__() call here on purpose - the subclass runs nn.Linear.__init__
		# FIRST, which already does it. calling it again would reset _modules and wipe lora_dropout.
		# (also: a bare super() here would resolve to nn.Linear under the MRO, not nn.Module,
		# and blow up on the missing in_features/out_features)
		
		self.rank = rank
		self.lora_alpha = lora_alpha
		self.lora_dropout = (
			nn.Dropout(lora_dropout) if lora_dropout > 0.0 else nn.Identity() # to keep the magnitude of output activations constant, we scale all the weights by 1/(1-p), so our output activation var = 1
		)
		self.use_rslora = use_rslora

		self.scaling = (
			self.lora_alpha / self.rank**0.5 if use_rslora else self.lora_alpha / self.rank
		)

	def load_pretrained_weights(self, state_dict):
		
		self.weight.data = state_dict["weight"] # load the pretrained matrix weights into the module
		if 'bias' in state_dict.keys():
			self.bias.data = state_dict["bias"]
		

	@abstractmethod
	def forward(self, x_in):
		...

	@abstractmethod
	def merge_weights(self):
		...

class LoraLinear(LoraBase, nn.Linear):

	def __init__(self, in_dim, out_dim, rank:int=8, lora_alpha:float=0.1, lora_dropout:float=0.1, use_rslora:bool=True, bias:bool=True): 
		
		# ORDER MATTERS. nn.Linear.__init__ internally calls nn.Module.__init__, which resets the
		# _modules dict - so it has to run BEFORE LoraBase registers lora_dropout, or that module
		# gets silently wiped and the forward dies with "no attribute 'lora_dropout'"
		nn.Linear.__init__(self, in_dim, out_dim, bias) # this is what creates self.weight (out_dim, in_dim) and self.bias

		# LoraBase's signature is (in_dim, out_dim, rank, lora_dropout, lora_alpha, use_rslora) -
		# pass by keyword so the dropout/alpha order can't silently swap
		LoraBase.__init__(self, in_dim, out_dim, rank=rank, lora_dropout=lora_dropout, lora_alpha=lora_alpha, use_rslora=use_rslora)
		
		self.weight.requires_grad = False # freeze the base - only A and B train
		if self.bias is not None:
			self.bias.requires_grad = False
		
		# rows are loaded from memory - since we will be multiplying in_dim by in_dim * rank, we want to arrange A
		# so that 1 x in_dim is loaded at a time. Similarly for B - we want to load it so 1 x rank is loaded at a time
		# this is just PyTorch's standard (out, in) layout: A maps in_dim -> rank, B maps rank -> out_dim
	
		self.lora_a = nn.Parameter(torch.empty(rank, in_dim)) # Lora A transpose
		self.lora_b = nn.Parameter(torch.zeros(out_dim, rank)) # Lora B transpose

		# A random, B zero -> B@A is zero at init, so the adapter is a no-op and the model starts
		# exactly at the pretrained output. gradients still reach A through B
		nn.init.kaiming_normal_(self.lora_a, nonlinearity='relu')

	def merge_weights(self):
		
		# (out, r) @ (r, in) -> (out, in). no transposes: both are already stored (out, in), so
		# their inner r dimensions line up directly. lora_a/lora_b are Parameters (raw tensors),
		# so there is no .weight to reach through
		lora = self.lora_b @ self.lora_a
		merged_weights = self.weight.data + lora * self.scaling

		state_dict = {"weight": merged_weights}
		if self.bias is not None: 
			state_dict["bias"] = self.bias.data

		merged_linear = nn.Linear(self.in_features, self.out_features, bias=self.bias is not None) # initialize a linear layer
		merged_linear.load_state_dict(state_dict) # load the state dict

		return merged_linear
	
	def forward(self, x_in):

		base = F.linear(x_in, self.weight, bias=self.bias) # x @ W^T + b
		# feed x through the rank bottleneck, NOT base - (..., in) -> (..., r) -> (..., out).
		# costs 2*d*r instead of the d*d of materializing B@A
		lora = F.linear(F.linear(self.lora_dropout(x_in), self.lora_a), self.lora_b) # [dropout(x_in) @ A.T] @ B.T

		return base + lora * self.scaling


layer = LoraLinear(in_dim=512, out_dim=256, rank=8)
x = torch.randn(4, 32, 512)
layer.eval() # disable dropout so the paths below are comparable

out = layer(x)
print(f"{tuple(x.shape)} -> {tuple(out.shape)}")

# B=0 at init, so the adapter contributes nothing yet - output must equal the frozen base layer
print("adapter is a no-op at init:", torch.allclose(out, F.linear(x, layer.weight, layer.bias)))

trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
print(f"trainable: {[n for n,p in layer.named_parameters() if p.requires_grad]} = {trainable:,} params vs {layer.weight.numel():,} frozen")

# after 'training', merging must reproduce the unmerged forward exactly
layer.lora_b.data = torch.randn_like(layer.lora_b) * 0.1
merged = layer.merge_weights()
print("merged == unmerged:", torch.allclose(layer(x), merged(x), atol=1e-5))

(4, 32, 512) -> (4, 32, 256)
adapter is a no-op at init: True
trainable: ['lora_a', 'lora_b'] = 6,144 params vs 131,072 frozen
merged == unmerged: True


In [ ]:
class LoRAEmbedding(nn.Embedding, LoraBase):
    def __init__(
        self,
        num_embeddings,
        embedding_dim,
        rank=8,
        lora_alpha=8,
        use_rslora=True,
        **kwargs,
    ):

        nn.Embedding.__init__(self, num_embeddings, embedding_dim, **kwargs)
        LoRALayerBase.__init__(
            self,
            rank=rank,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            use_rslora=use_rslora,
        )

        self.weight.requires_grad = False  # freeze the weight matrix
        print(self.weight.shape)

        self.lora_A = nn.Parameter(torch.zeros(num_embeddings, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, embedding_dim))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def _merge_weights(self):
        # element-wise addition and multiplication
        merged_weights = self.weight.data + (self.lora_A @ self.lora_B).T * self.scaling

        state_dict = {"weight": merged_weights}

        merged_embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        merged_embedding.load_state_dict(state_dict)

        return merged_embedding

    def forward(self, x):

        orig_layer_out = F.embedding(
            input=x,
            weight=self.weight,
            padding_idx=self.padding_idx,
            max_norm=self.max_norm,
            norm_type=self.norm_type,
            scale_grad_by_freq=self.scale_grad_by_freq,
            sparse=self.sparse,
        )

        low_rank_A_output = F.embedding(
            input=x,
            weight=self.lora_A,
            padding_idx=self.padding_idx,
            max_norm=self.max_norm,
            norm_type=self.norm_type,
            scale_grad_by_freq=self.scale_grad_by_freq,
            sparse=self.sparse,
        )

        low_rank_output = (low_rank_A_output @ self.lora_B) * self.scaling

        output = orig_layer_out + low_rank_output
        return output

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'Qwen/Qwen3.5-0.8B'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype='auto', device_map='auto')

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 913.15it/s] 


<generator object Module.named_children at 0x117658040>


In [ ]:
from dataclasses import dataclass, field
from typing import Optional, Union, Literal

@dataclass
class LoraConfig:
	rank: int = 8
	lora_alpha: int = 8
	lora_dropout: float = 0.0
	use_rslora: bool = True
	target_modules: Optional[Union[list[str], str]] = field(default_factory=list) # creates default array on initialization as a new object in memory
	exclude_modules: Optional[Union[list[str], str]] = field(default_factory=list)
	bias: Literal["none", "all", "lora_only"] = "none"

class LoraModel(nn.Module):

	def __init__(self, model, cfg:LoraConfig):

		super().__init__()

		self.model = model
		self.cfg = cfg

		self._apply_lora()
		self._freeze_base()

	def _exclude_module_name_check(self, name):
		return any(ex in name for ex in self.cfg.exclude_modules)

	def _target_module_name_check(self, name):
		return any(t in name for t in self.cfg.target_modules)

	def _apply_lora(self):

		# finished training, time to merge weights

		# list() snapshots the tree before we start mutating it
		for path, child in list(self.model.named_modules()):

			# guard clauses: skip anything that isn't a target, so the path-splitting
			# below only runs for layers we're actually swapping
			if not isinstance(child, nn.Linear):
				continue
			if not self._target_module_name_check(path):
				continue
			if self._exclude_module_name_check(path):
				continue

			# "model.layers.0.self_attn.q_proj" -> parent object + leaf attribute name.
			parent_path, _, leaf = path.rpartition('.') # searches for hte last dot on the right side and splits the string into a 3 part tuple
			parent = self.model.get_submodule(parent_path) if parent_path else self.model

			lora_layer = LoraLinear(
				child.in_features, child.out_features,
				rank=self.cfg.rank, lora_alpha=self.cfg.lora_alpha,
				lora_dropout=self.cfg.lora_dropout, use_rslora=self.cfg.use_rslora,
				bias=child.bias is not None,
			)
			lora_layer.load_pretrained_weights(child.state_dict())

			lora_layer.to(device=child.weight.device, dtype=child.weight.dtype) # moves the lora weight matrix to bf16
			setattr(parent, leaf, lora_layer)

	def _freeze_base(self):

		for name, param in self.model.named_parameters():
			param.requires_grad = name.endswith("lora_a") or name.endswith("lora_b") # sets everything as False unless ends with lora_a or lora_b

		if self.cfg.bias == "all":
			for name, param in self.model.named_parameters():
				if name.endswith("bias"):
					param.requires_grad = True
		elif self.cfg.bias == "lora_only":
			for module in self.model.modules():
				if isinstance(module, LoraLinear) and module.bias is not None:
					module.bias.requires_grad = True

	def print_trainable_parameters(self):

		trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
		total = sum(p.numel() for p in self.model.parameters())
		print(f"trainable: {trainable:,} / {total:,} = {100*trainable/total:.4f}%")

	def __getattr__(self, name):
		# nn.Module has its own __getattr__ for params/buffers/submodules - try that first,
		# then fall through to the wrapped model for .generate(), .config, etc.
		try:
			return super().__getattr__(name)
		except AttributeError:
			return getattr(self.model, name)

	def forward(self, *args, **kwargs):
		return self.model(*args, **kwargs)

QWEN_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

config = LoraConfig(target_modules=QWEN_TARGETS)
lora_model = LoraModel(model, config)
lora_model.print_trainable_parameters()

wrapped = [n for n, m in lora_model.model.named_modules() if isinstance(m, LoraLinear)]
print(f"wrapped {len(wrapped)} layers, e.g. {wrapped[:3]}")
print("all trainable params are adapters:", all(
	n.endswith(("lora_a", "lora_b")) for n, p in lora_model.named_parameters() if p.requires_grad
))


trainable: 3,194,880 / 755,587,904 = 0.4228%
wrapped 96 layers, e.g. ['model.layers.0.mlp.gate_proj', 'model.layers.0.mlp.up_proj', 'model.layers.0.mlp.down_proj']
all trainable params are adapters: True


**Understanding multi-class inheritance**

- super().**init**() on a base class initializes the next parent class(e.g. on `Vehicle` in `class Car(Vehicle, Wheeled):` rather than Motor like `class Vehicle(Motor):`)

- this initializes the next parent class that is Wheeled.**init**() rather than the parent class of Vehicle

- you can imagine classes in layers, you have Car (child class), Vehicle & Wheeled (parent classes), then Motor (grandparent class)

- The parent classes are initialized -> then grandparent -> ...


In [ ]:
LoraLinear.__mro__ # method resolution order

(__main__.LoraLinear,
 __main__.LoraBase,
 torch.nn.modules.linear.Linear,
 torch.nn.modules.module.Module,
 object)

In [31]:
lora_model.named_parameters() # name, tensors

lora_model.named_children() # name, top-level blocks / containers

lora_model.named_modules() # unpacks entire network down to activation functions

<generator object Module.named_modules at 0x1788fdf10>